In [1]:
import osmnx as ox
import numpy as np

import os
import time
from pathlib import Path



In [ ]:
start_point = 'Hof Schönenberg 2, 4133 Pratteln'
end_point = 'Dompl. 16, 4144 Arlesheim'

start_time = 1713700000

#speed [m/s]
#rainsensibility['low'|'medium'|'high']
#routingmodel ['einfach'|'funktion'|'rerouting'|'all']
#modus['live'|'demo']

In [21]:
from datetime import datetime
now = datetime.now() 

now

datetime.datetime(2026, 4, 23, 13, 44, 32, 815950)

In [ ]:
from get_graph import _parse_point, get_boundingbox_from_points, get_graph_cached


start_point = _parse_point(start_point)
end_point = _parse_point(end_point)

bbox = get_boundingbox_from_points(start_point, end_point, 4)

G = get_graph_cached(bbox)


In [ ]:
def get_nc_file(start_time, parentfolder='nc_folder', valid_time=33*3600):
    """
    Gibt das gültige NC-File mit dem neuesten Zeitstempel zurück.
    
    Parameter
    ----------
    start_time : float
        Referenz-Zeitstempel (z.B. aktuelle Zeit mit time.time())
    parentfolder : str, optional
        Pfad zum Ordner mit den NC-Files (default: 'nc_folder')
    valid_time : int, optional
        Gültigkeitsdauer in Sekunden (default: 33 Stunden = 118800 Sekunden)
    
    Returns
    -------
    str or None
        Pfad zum neuesten gültigen NC-File, oder None wenn keines gültig ist
    """
    
    # Sicherstellen, dass der Ordner existiert
    if not os.path.exists(parentfolder):
        raise FileNotFoundError(f"Ordner '{parentfolder}' existiert nicht")
    valid_files = []
    
    # Alle .nc-Dateien im Ordner durchsuchen
    for filename in os.listdir(parentfolder):
        if filename.endswith('.nc'):

            # Zeitstempel aus dem Dateinamen extrahieren
            # Erwartet Format: "{timestamp}.nc" z.B. "1234567890.nc"
            try:
                # Alles vor .nc entfernen und zu Integer konvertieren
                timestamp_str = filename[:-3]  # Entfernt ".nc"
                file_timestamp = int(timestamp_str)
            except ValueError:
                # Wenn Zeitstempel nicht parsbar ist, Datei überspringen
                continue
        
            # Prüfen ob Datei noch gültig ist
            age = start_time - file_timestamp
            if 0 <= age <= valid_time:
                full_path = os.path.join(parentfolder, filename)
                valid_files.append((file_timestamp, full_path))
            
    
    # Neuste Datei oder None
    if not valid_files:
        return None
    else:
        newest_files = max(valid_files, key=lambda x: x[0])
        return newest_files[1]

In [35]:
file = get_nc_file(start_time=start_time)

print(file)

nc_folder\1713650000.nc


In [ ]:
def get_nearest_node(G, point):
    """
    Gibt die nächste Node zurück, egal ob der Point als Adresse oder Koordinate definiert wurde.

    Parameter
    ----------
    G : networkx.MultiDiGraph
        OSMnx-Graph
    point : str | tuple | list | np.ndarray
        Adresse (String) ODER Koordinaten im Format (lat, lon)

    Returns
    -------
    nearest_node : int
        ID der nächstgelegenen Node im Graph
    """

    # Fall 1: Adresse (String)
    if isinstance(point, str):
        lat, lon = ox.geocode(point)

    # Fall 2: Koordinaten
    elif isinstance(point, (list, tuple, np.ndarray)):
        if len(point) != 2:
            raise ValueError(f"{point} muss genau 2 Werte enthalten (lat, lon)")
        lat, lon = float(point[0]), float(point[1])

    else:
        raise ValueError(f"{point} muss eine Adresse oder Koordinaten (lat, lon) sein")

    # Sicherheitscheck
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        raise ValueError(f"Ungültige Koordinaten: ({lat}, {lon})")

    # Achtung: X = longitude, Y = latitude
    nearest_node = ox.distance.nearest_nodes(G, X=lon, Y=lat)

    return nearest_node

In [ ]:
import osmnx as ox

address = "Bahnhofstrasse 1, Zürich"
point = ox.geocode(address)  # (lat, lon)